# 🧹 Smart CV Analyzer: Tahap 2 - Data Cleaning & Tokenisasi

Pada tahap ini, kita akan membersihkan file `cv_sentences_raw.csv` dari *noise* hasil ekstraksi PDF. 

Langkah-langkah yang akan dilakukan:
1. **Cleaning:** Menghapus karakter aneh seperti `(cid:127)` (bullet point dari PDF).
2. **PII Removal:** Membuang baris yang mengandung data privasi (Email, Nomor HP, atau Link Profil) untuk keamanan data.
3. **Tokenisasi:** Memecah kalimat utuh menjadi daftar kata (*list of tokens*) di mana tanda baca dipisah dari kata.
4. **JSONL Formatting:** Menyimpan hasil akhir ke format `.jsonl` dengan memberikan label bawaan `"O"` (Outside) untuk semua kata, sehingga siap untuk dianotasi secara manual.

## Load Data Awal

In [1]:
import pandas as pd
import re
import json

# Load data mentah dari Tahap 1
df_raw = pd.read_csv("cv_sentences_raw.csv")
print(f"📊 Total data mentah awal: {len(df_raw)} baris")

📊 Total data mentah awal: 1179 baris


## 1. Fungsi Pembersihan Teks (*Data Cleaning*)
Fungsi `clean_cv_text` digunakan untuk merapikan karakter di dalam teks, sedangkan `is_valid_sentence` bertugas sebagai filter untuk membuang baris yang tidak memenuhi syarat (terlalu pendek atau mengandung informasi pribadi).

In [2]:
def clean_cv_text(text):
    text = str(text)
    # Hapus karakter aneh dari PDF seperti (cid:127)
    text = re.sub(r'\(cid:\d+\)', '', text)
    # Hapus sisa bullet points klasik (seperti -, •, *) di awal kalimat
    text = re.sub(r'^[•\-\*]\s*', '', text)
    
    # Normalisasi Unicode em-dash (\u2014) dan en-dash (\u2013) menjadi strip biasa (-)
    text = text.replace('\u2014', '-').replace('\u2013', '-')
    text = text.replace('—', '-').replace('–', '-') # Berjaga-jaga jika terbaca sebagai karakter aslinya
    
    # Hapus spasi berlebih dan rapikan
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_valid_sentence(text):
    # 1. Buang kalimat yang terlalu pendek (kurang dari 3 kata)
    if len(text.split()) < 3:
        return False
    # 2. Buang baris yang berisi Email (Privasi)
    if re.search(r'[\w\.-]+@[\w\.-]+', text):
        return False
    # 3. Buang baris yang berisi Nomor Telepon/HP (Privasi)
    if re.search(r'(\+62|08)\s?\d{3,}', text):
        return False
    # 4. Buang baris yang isinya URL profil (LinkedIn/GitHub)
    if "github.com" in text.lower() or "linkedin.com" in text.lower():
        return False
        
    return True

# Terapkan pembersihan karakter terlebih dahulu
df_raw['clean_text'] = df_raw['text'].apply(clean_cv_text)

# Terapkan filter baris yang valid
df_clean = df_raw[df_raw['clean_text'].apply(is_valid_sentence)].copy()

print(f"✨ Total data setelah dibersihkan (siap ditokenisasi): {len(df_clean)} baris")

✨ Total data setelah dibersihkan (siap ditokenisasi): 980 baris


### 2. Tokenisasi dan Ekspor ke JSONL
Teks yang sudah bersih kini akan dipecah. Tanda baca seperti koma (`,`) dan titik (`.`) akan dipisahkan menjadi token tersendiri agar model Machine Learning bisa mengenali batas kata dengan akurat.

In [3]:
def tokenize_text(text):
    # Regex ini memisahkan kata (alfanumerik) dan tanda baca.
    # Contoh: "Python, SQL." -> ["Python", ",", "SQL", "."]
    tokens = re.findall(r'\w+|[^\w\s]', text)
    return tokens

# Buat kolom baru berisi list of tokens
df_clean['tokens'] = df_clean['clean_text'].apply(tokenize_text)

# Siapkan list untuk menampung format JSONL
jsonl_data = []

for tokens in df_clean['tokens']:
    # Buat label "O" (Outside) default sebanyak jumlah token di kalimat tersebut
    default_tags = ["O"] * len(tokens)
    
    jsonl_data.append({
        "tokens": tokens,
        "ner_tags": default_tags
    })

# Simpan ke file JSONL
output_jsonl = "cv_dataset_ready_to_tag.jsonl"
with open(output_jsonl, "w", encoding="utf-8") as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + "\n")

print(f"✅ Tahap 2 Selesai! Data berhasil diekspor ke: {output_jsonl}")

# Tampilkan preview format JSONL untuk 1 kalimat pertama
print("\n🔍 Preview format JSONL:")
print(json.dumps(jsonl_data[0], indent=2))

✅ Tahap 2 Selesai! Data berhasil diekspor ke: cv_dataset_ready_to_tag.jsonl

🔍 Preview format JSONL:
{
  "tokens": [
    "Data",
    "engineer",
    "yang",
    "berpengalaman",
    "dalam",
    "membangun",
    "dan",
    "mengelola",
    "pipeline",
    "data",
    "skala",
    "besar"
  ],
  "ner_tags": [
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O"
  ]
}
